# CIFAR-10 Image Classification

A simple CNN project using PyTorch.

Dataset folder: `./data/`

Run the cells from top to bottom. A GPU is recommended.

## 1. Import libraries and choose device

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

batch_size = 64
classes = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

## 2. Load CIFAR-10 and show images

In [ ]:
simple_transform = transforms.ToTensor()

data = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=simple_transform
)

# Show 10 images from each class
fig, axes = plt.subplots(10, 10, figsize=(12, 12))
count = [0] * 10

for image, label in data:
    if count[label] < 10:
        axes[label, count[label]].imshow(image.permute(1, 2, 0))
        axes[label, count[label]].axis("off")
        if count[label] == 0:
            axes[label, count[label]].set_ylabel(classes[label])
        count[label] += 1

    if all(x == 10 for x in count):
        break

plt.show()

## 3. Class balance and pixel values

In [ ]:
labels = np.array(data.targets)

plt.figure(figsize=(8, 4))
plt.bar(classes, np.bincount(labels))
plt.xticks(rotation=45)
plt.title("Number of images in each class")
plt.show()

# Check pixel values from one image
image, label = data[0]
print("Minimum pixel value:", image.min().item())
print("Maximum pixel value:", image.max().item())

plt.hist(image.numpy().flatten(), bins=30)
plt.title("Pixel value distribution")
plt.show()

## 4. Preprocessing and data augmentation

In [ ]:
# ToTensor changes pixel values to the range 0 to 1.
# Training images also get random augmentation.

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

test_transform = transforms.ToTensor()

train_aug = datasets.CIFAR10(
    "./data", train=True, download=True, transform=train_transform
)

train_plain = datasets.CIFAR10(
    "./data", train=True, download=True, transform=test_transform
)

test_data = datasets.CIFAR10(
    "./data", train=False, download=True, transform=test_transform
)

# One-hot encoding example
label = torch.tensor(3)
print("Original label:", label.item())
print("One-hot label:", nn.functional.one_hot(label, num_classes=10))

## 5. Create data loaders

In [ ]:
# Use some training images for validation
train_indices = list(range(45000))
val_indices = list(range(45000, 50000))

train_data = Subset(train_aug, train_indices)
val_data = Subset(train_plain, val_indices)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 6. Build the baseline CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = CNN().to(device)
print(model)

## 7. Model parameters

In [ ]:
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

## 8. Train the model

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

epochs = 50
best_loss = float("inf")
patience = 15
wait = 0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(epochs):

    # Training
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss / total
    val_acc = correct / total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # Save the best model
    if val_loss < best_loss:
        best_loss = val_loss
        wait = 0
        torch.save(model.state_dict(), "best_cifar10_model.pth")
    else:
        wait += 1

    # Stop if validation loss does not improve
    if wait == patience:
        print("Early stopping")
        break

model.load_state_dict(torch.load("best_cifar10_model.pth", map_location=device))

## 9. Plot accuracy and loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train")
plt.plot(val_accuracies, label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## 10. Test the model

In [ ]:
model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)

        predictions = outputs.argmax(1).cpu().numpy()
        all_predictions.extend(predictions)
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

accuracy = (all_labels == all_predictions).mean()
print("Test accuracy:", round(accuracy * 100, 2), "%")

## 11. Confusion matrix and class accuracy

In [ ]:
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

for i in range(10):
    class_acc = cm[i, i] / cm[i].sum()
    print(classes[i], "accuracy:", round(class_acc * 100, 2), "%")

# Show the class pairs that are most often confused
cm_no_diagonal = cm.copy()
np.fill_diagonal(cm_no_diagonal, 0)

for _ in range(5):
    true_class, predicted_class = np.unravel_index(
        cm_no_diagonal.argmax(), cm_no_diagonal.shape
    )
    print(classes[true_class], "mistaken for", classes[predicted_class])
    cm_no_diagonal[true_class, predicted_class] = 0

## 12. Show some wrong predictions

In [ ]:
wrong = np.where(all_labels != all_predictions)[0]

plt.figure(figsize=(12, 6))
for i, index in enumerate(wrong[:10]):
    image, _ = test_data[index]

    plt.subplot(2, 5, i + 1)
    plt.imshow(image.permute(1, 2, 0))
    plt.title(
        "True: " + classes[all_labels[index]] +
        "\nPred: " + classes[all_predictions[index]],
        fontsize=8
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 13. Show filters from the first convolution layer

In [ ]:
filters = model.conv1.weight.detach().cpu()

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    filter_image = filters[i].permute(1, 2, 0)
    filter_image = (filter_image - filter_image.min()) / (
        filter_image.max() - filter_image.min() + 1e-8
    )
    ax.imshow(filter_image)
    ax.axis("off")

plt.show()

## 14. Show feature maps

In [ ]:
image, label = test_data[0]
model.eval()

with torch.no_grad():
    features = torch.relu(model.conv1(image.unsqueeze(0).to(device)))
    features = features.squeeze(0).cpu()

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(features[i], cmap="gray")
    ax.axis("off")

plt.suptitle("Feature maps from the first convolution layer")
plt.show()

## 15. Simple Grad-CAM

In [ ]:
# Grad-CAM shows which image areas influenced the prediction.
model.eval()

image, true_label = test_data[0]
input_image = image.unsqueeze(0).to(device)

feature_maps = None

def save_features(module, input, output):
    global feature_maps
    feature_maps = output
    feature_maps.retain_grad()

hook = model.conv1.register_forward_hook(save_features)

output = model(input_image)
predicted_label = output.argmax(1).item()

model.zero_grad()
output[0, predicted_label].backward()

# Average the gradients and combine them with the feature maps
weights = feature_maps.grad.mean(dim=(2, 3), keepdim=True)
heatmap = (weights * feature_maps).sum(dim=1).squeeze()
heatmap = torch.relu(heatmap).detach().cpu().numpy()
heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

hook.remove()

plt.imshow(image.permute(1, 2, 0))
plt.imshow(heatmap, cmap="jet", alpha=0.4)
plt.title("Grad-CAM - True: " + classes[true_label] +
          ", Predicted: " + classes[predicted_label])
plt.axis("off")
plt.show()

## Conclusion

The baseline CNN was trained to classify CIFAR-10 images. We checked its accuracy, confusion matrix, wrong predictions, filters, feature maps and Grad-CAM heatmap.